# NOTS-NIDS: Setup Notebook
**Nash-Optimized Topological Shields**

Run this notebook first to set up the environment on Google Colab or Kaggle.

In [ ]:
# Cell 1 — Mount Drive and check GPU
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab')

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — switch to GPU runtime'
print(f'GPU: {gpu_name}')

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
%%bash
pip install -q ripser persim pot umap-learn cvxpy scikit-learn pandas numpy \
    matplotlib seaborn joblib torch torchvision tqdm memory-profiler
# Try cuML (GPU UMAP) — only works on Colab with T4/A100
pip install -q cudf-cu11 cuml-cu11 --extra-index-url=https://pypi.nvidia.com 2>/dev/null || \
    echo 'cuML not available, using CPU UMAP'

In [ ]:
# Cell 3 — Setup repo path
import sys
import os

if IN_COLAB:
    REPO_DIR = '/content/drive/MyDrive/nots_nids'
else:
    REPO_DIR = os.path.dirname(os.getcwd())

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from config import Config
cfg = Config()
print(f'Config loaded. Results will save to: {cfg.RESULTS_DIR}')

In [ ]:
# Cell 4 — Sanity check: run a tiny persistence diagram
import numpy as np
from ripser import ripser
from persim import wasserstein

np.random.seed(42)
points = np.random.rand(50, 5)
dgms = ripser(points, maxdim=1)['dgms']
print(f'H0 features: {len(dgms[0])}, H1 features: {len(dgms[1])}')
print('Ripser working correctly.')

# Quick Wasserstein test
points2 = np.random.rand(50, 5)
dgms2 = ripser(points2, maxdim=1)['dgms']
w = wasserstein(dgms[1], dgms2[1])
print(f'Wasserstein distance (H1): {w:.4f}')
print('\n✓ All dependencies verified!')